# День 1. От текста к вектору: базовые методы NLP

**План на 4 пары по 90 минут.** Целевая аудитория — студенты технических специальностей. Сквозной материал: задачи по теории вероятностей и математической статистике.

В конце дня студент умеет подготовить технический текст, построить BoW и TF–IDF-векторы, вычислить косинусную близость и выполнить базовый поиск похожих задач.

**Пары:** 1) текст и токенизация; 2) словари, n-граммы и частоты; 3) векторизация BoW/TF–IDF; 4) косинусная близость и поиск по корпусу.

Установка библиотек: `python -m pip install scikit-learn matplotlib`.

## Пара 1. Технический текст как данные

### 1.1. Корпус задач

При обработке математических текстов важно не уничтожить обозначения, числа, знаки сравнения и отрицания. Например, «не отвергать гипотезу» и «отвергать гипотезу» имеют противоположный смысл.

In [ ]:
documents = [
    "Найти вероятность противоположного события, если P(A) = 0.3.",
    "Для биномиальной случайной величины X ~ Bin(n, p) найти математическое ожидание.",
    "Проверить гипотезу H0: mu = 0 при уровне значимости alpha = 0.05.",
    "Построить доверительный интервал для среднего по выборке.",
    "Вычислить медиану и межквартильный размах выборки.",
    "Найти коэффициент корреляции между двумя случайными величинами.",
    "Для нормальной величины X ~ N(mu, sigma^2) вычислить P(X < a).",
    "Оценить вероятность события методом Монте-Карло."
]
for i, text in enumerate(documents, 1):
    print(f"{i}. {text}")

### 1.2. Очистка и нормализация

Нормализация пробелов и регистра полезна для поиска. Но универсальная очистка «оставить только буквы» может потерять важные части условия. Сравните два варианта токенизации.

In [ ]:
import re
import unicodedata

def normalize(text):
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

def tokenize_words(text):
    return re.findall(r"[а-яёa-z]+", normalize(text))

def tokenize_technical(text):
    pattern = r"[a-zа-яё]+(?:-[a-zа-яё]+)*|\d+(?:[.,]\d+)?|[=<>~]+|[αβμσ²]+"
    return re.findall(pattern, normalize(text))

sample = "H0: mu = 0; p-value < 0.05; X ~ N(mu, sigma^2)"
print("Только слова:", tokenize_words(sample))
print("Техническая токенизация:", tokenize_technical(sample))

### 1.3. Контекст и отрицания

Стоп-слова нельзя удалять механически. В техническом тексте слова «не», «менее», «больше», «против» могут изменить смысл задачи. Очистка должна учитывать назначение системы: поиск похожих формулировок, классификация или извлечение условия.

In [ ]:
examples = [
    "Гипотеза отвергается при p-value < 0.05",
    "Гипотеза не отвергается при p-value >= 0.05",
    "Найти вероятность не менее трех успехов",
    "Найти вероятность не более трех успехов"
]
for text in examples:
    print(tokenize_technical(text))

### Задания к паре 1

1. Сравните токены для (P(X < a)) и (P(X > a)). Что должно сохраниться?
2. Почему опасно удалять слово «не»?
3. Напишите правило токенизации для обозначений `p-value`, `x_1`, `0.05`, `>=`.
4. Придумайте два примера, где математический знак меняет ответ задачи.

## Пара 2. Словарь, частоты и n-граммы

### 2.1. Частотный анализ

Частоты помогают исследовать корпус и выделять термины, но не отражают автоматически смысл или важность слова.

In [ ]:
from collections import Counter

tokens_by_doc = [tokenize_words(doc) for doc in documents]
all_tokens = [token for row in tokens_by_doc for token in row]
freq = Counter(all_tokens)

print("Документов:", len(documents))
print("Токенов:", len(all_tokens))
print("Уникальных токенов:", len(set(all_tokens)))
print(freq.most_common(20))

### 2.2. Визуализация частот

In [ ]:
import matplotlib.pyplot as plt

top = freq.most_common(15)
plt.figure(figsize=(10, 4))
plt.bar([x[0] for x in top], [x[1] for x in top])
plt.xticks(rotation=55, ha="right")
plt.ylabel("Частота")
plt.title("Частоты слов в корпусе задач")
plt.tight_layout()
plt.show()

### 2.3. N-граммы: почему словосочетания важны

«Случайная величина», «нулевая гипотеза», «доверительный интервал» — устойчивые термины. Биграммы и триграммы сохраняют часть локального контекста, хотя не понимают значение фразы целиком.

In [ ]:
def ngrams(tokens, n=2):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

for doc in documents[:4]:
    toks = tokenize_words(doc)
    print("\n", doc)
    print("Биграммы:", ngrams(toks, 2))
    print("Триграммы:", ngrams(toks, 3))

### Задания к паре 2

1. Найдите биграммы, указывающие на тему «проверка гипотез».
2. Сравните частоту слова «интервал» и фразы «доверительный интервал».
3. Почему абсолютные частоты нельзя напрямую сравнивать для корпусов разного размера?
4. Добавьте 5 задач по одной теме и проверьте, какие термины стали чаще встречаться.

## Пара 3. Векторизация: Bag of Words и TF–IDF

### 3.1. Bag of Words

Bag of Words (BoW) представляет каждый документ частотами слов общего словаря. Порядок слов теряется, поэтому две фразы с одинаковыми словами могут получить одинаковое представление.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

toy = [
    "случайная величина имеет нормальное распределение",
    "нормальное распределение имеет случайная величина",
    "проверить нулевую гипотезу по выборке",
    "построить доверительный интервал по выборке"
]
count_vec = CountVectorizer(token_pattern=r"(?u)\b[а-яёА-ЯЁa-zA-Z]+\b")
X_count = count_vec.fit_transform(toy)
print("Признаки:", count_vec.get_feature_names_out().tolist())
print("Матрица BoW:\n", X_count.toarray())

### 3.2. TF–IDF

TF–IDF повышает относительный вес терминов, характерных для документа, и снижает вес терминов, встречающихся во многих документах. Результат зависит от корпуса и параметров векторизатора.

In [ ]:
tfidf_vec = TfidfVectorizer(
    token_pattern=r"(?u)\b[а-яёА-ЯЁa-zA-Z]+\b",
    ngram_range=(1, 2)
)
X_tfidf = tfidf_vec.fit_transform(documents)
print("Размер матрицы:", X_tfidf.shape)
print("Первые признаки:", tfidf_vec.get_feature_names_out()[:25])
print("Вектор первого документа:\n", X_tfidf[0].toarray().round(3))

### 3.3. Разреженность и интерпретация

В корпусе каждый документ использует лишь небольшую часть общего словаря, поэтому матрица документов и признаков обычно разреженная. TF–IDF — не понимание смысла: синонимы могут оказаться далеко друг от друга, если не имеют общих слов.

In [ ]:
import numpy as np

dense = X_tfidf.toarray()
total = dense.size
nonzero = np.count_nonzero(dense)
print(f"Ненулевых элементов: {nonzero}/{total}")
print(f"Доля нулей: {1 - nonzero/total:.1%}")

### Задания к паре 3

1. Составьте BoW-векторы для «бросить монету три раза» и «бросить кубик три раза».
2. Почему BoW не кодирует порядок слов?
3. Сравните TF–IDF для общего слова «найти» и специализированного термина «биномиальное».
4. Как изменится матрица при добавлении новых документов?

## Пара 4. Косинусная близость и поиск похожих задач

### 4.1. Косинусная близость

Для ненулевых векторов (x,y):
\[
\cos(x,y)=\frac{x^\top y}{\|x\|_2\|y\|_2}.
\]
Значение выше означает более близкое направление в пространстве признаков. В TF–IDF это в первую очередь лексическая близость, а не гарантия эквивалентности математического смысла.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

A = np.array([[1, 1, 0]])
B = np.array([[1, 0, 1]])
C = np.array([[2, 2, 0]])
print("cos(A,B):", cosine_similarity(A, B)[0, 0])
print("cos(A,C):", cosine_similarity(A, C)[0, 0])

### 4.2. Поисковая функция

Запрос преобразуется тем же векторизатором, что и документы корпуса. Затем вычисляется сходство запроса с каждым документом и возвращается топ результатов. Важно: новый запрос не должен повторно обучать векторизатор.

In [ ]:
search_vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b[а-яёА-ЯЁa-zA-Z]+\b",
    ngram_range=(1, 2),
    sublinear_tf=True
)
X = search_vectorizer.fit_transform(documents)

def search(query, top_k=4):
    q = search_vectorizer.transform([query])
    scores = cosine_similarity(q, X).ravel()
    order = np.argsort(scores)[::-1][:top_k]
    return [(round(float(scores[i]), 3), documents[i]) for i in order]

for score, text in search("вероятность нормальной случайной величины"):
    print(score, text)

### 4.3. Проверка качества выдачи вручную

Попробуйте запросы:
- «вероятность не менее двух успехов»;
- «проверить статистическую гипотезу»;
- «оценить разброс данных»;
- «найти среднее по наблюдениям».

Запишите, попадает ли релевантная задача в топ-1 и топ-3. TF–IDF может не найти синонимы и перефразировки; это мотивирует семантические эмбеддинги, которые изучаются во второй день.

### Итог дня 1

Создайте корпус из 15 формулировок задач по терверу или матстатистике. Реализуйте цепочку: очистка → токенизация → TF–IDF → поиск по косинусной близости. Подготовьте 5 запросов и вручную оцените релевантность топ-3 результатов. Сохраните корпус, запросы и наблюдения для следующего дня.